## Reranking Problem & Setup

In [1]:
import os
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
customer_features = pd.read_csv(
    "../data/customer_features.csv"
)

product_features = pd.read_csv(
    "../data/product_features.csv"
)

order_analytics = pd.read_csv(
    "../data/order_analytics.csv"
)

metadata = pd.read_csv(
    "../data/commerceiq_document_metadata.csv"
)

embeddings = np.load(
    "../data/commerceiq_embeddings.npy"
)

print("Customer features:", customer_features.shape)
print("Product features:", product_features.shape)
print("Order analytics:", order_analytics.shape)
print("Metadata:", metadata.shape)
print("Embeddings:", embeddings.shape)

Customer features: (4335, 13)
Product features: (3918, 11)
Order analytics: (19865, 9)
Metadata: (28118, 4)
Embeddings: (28118, 384)


In [3]:
embedding_dim = embeddings.shape[1]

rerank_index = faiss.IndexFlatIP(
    embedding_dim
)

rerank_index.add(
    embeddings.astype("float32")
)

print("Embedding dimension:", embedding_dim)
print("FAISS index size:", rerank_index.ntotal)

Embedding dimension: 384
FAISS index size: 28118


In [4]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model loaded."
)

print(
    "Embedding dimension:",
    embedding_model.get_embedding_dimension()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384


In [5]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-encoder reranker loaded successfully.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

e:\commerceiq\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sadiya Sajid\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder reranker loaded successfully.


In [7]:
print("SETUP VALIDATION")
print("=" * 40)

print("Metadata rows:", len(metadata))
print("Embedding rows:", len(embeddings))
print("FAISS vectors:", rerank_index.ntotal)
print(
    "Embedding dimensions:",
    embeddings.shape[1]
)

print(
    "Reranker:",
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("\nSetup status: PASS")

SETUP VALIDATION
Metadata rows: 28118
Embedding rows: 28118
FAISS vectors: 28118
Embedding dimensions: 384
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

Setup status: PASS


## Retrieve Candidate Set

In [32]:
def retrieve_candidates(query, source, top_k=20):
    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # Select documents from the correct source
    source_documents = business_documents[
        business_documents["Source"] == source
    ].copy()

    # Encode only those candidate documents
    document_embeddings = embedding_model.encode(
        source_documents["Text"].tolist(),
        normalize_embeddings=True
    ).astype("float32")

    # Calculate cosine similarity
    scores = np.dot(
        document_embeddings,
        query_embedding[0]
    )

    # Add scores and rank
    source_documents["Semantic_Score"] = scores

    source_documents = (
        source_documents
        .sort_values("Semantic_Score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    source_documents["Rank"] = range(
        1,
        len(source_documents) + 1
    )

    return source_documents[
        [
            "Rank",
            "Semantic_Score",
            "Source",
            "Document_ID",
            "Text"
        ]
    ]

In [33]:
customer_candidates = retrieve_candidates(
    "Which customer segment generates the most revenue?",
    source="customer_segment",
    top_k=20
)

customer_candidates.head(10)

,Rank,Semantic_Score,Source,Document_ID,Text
0,1,0.764751,customer_segment,customer_segment_Medium,"Customer revenue segment Medium contains 1,084..."
1,2,0.756317,customer_segment,customer_segment_High,"Customer revenue segment High contains 1,083 c..."
2,3,0.733111,customer_segment,customer_segment_Very High,"Customer revenue segment Very High contains 1,..."
3,4,0.632962,customer_segment,customer_segment_Low,"Customer revenue segment Low contains 1,084 cu..."


In [34]:
product_candidates = retrieve_candidates(
    "Which products are strongest revenue contributors?",
    source="product_summary",
    top_k=20
)

product_candidates.head(10)

,Rank,Semantic_Score,Source,Document_ID,Text
0,1,0.465162,product_summary,product_summary_22197,Product revenue analysis: SMALL POPCORN HOLDER...
1,2,0.443205,product_summary,product_summary_23843,"Product revenue analysis: PAPER CRAFT , LITTLE..."
2,3,0.435921,product_summary,product_summary_84879,Product revenue analysis: ASSORTED COLOUR BIRD...
3,4,0.418270,product_summary,product_summary_23166,Product revenue analysis: MEDIUM CERAMIC TOP S...
4,5,0.415534,product_summary,product_summary_21137,Product revenue analysis: BLACK RECORD COVER F...
5,6,0.414155,product_summary,product_summary_47566,Product revenue analysis: PARTY BUNTING (Stock...
6,7,0.409420,product_summary,product_summary_22960,Product revenue analysis: JAM MAKING SET WITH ...
7,8,0.403034,product_summary,product_summary_22086,Product revenue analysis: PAPER CHAIN KIT 50'S...
8,9,0.401836,product_summary,product_summary_22386,Product revenue analysis: JUMBO BAG PINK POLKA...
9,10,0.393035,product_summary,product_summary_23084,Product revenue analysis: RABBIT NIGHT LIGHT (...


In [35]:
retention_candidates = retrieve_candidates(
    "Which customer groups are at risk of becoming inactive?",
    source="activity_summary",
    top_k=20
)

retention_candidates.head(10)

,Rank,Semantic_Score,Source,Document_ID,Text
0,1,0.527435,activity_summary,activity_summary_Inactive,Customer activity segment Inactive contains 86...
1,2,0.510447,activity_summary,activity_summary_Recently Inactive,Customer activity segment Recently Inactive co...
2,3,0.422379,activity_summary,activity_summary_At Risk,Customer activity segment At Risk contains 586...
3,4,0.392397,activity_summary,activity_summary_Active,"Customer activity segment Active contains 1,64..."


In [36]:
print("CANDIDATE RETRIEVAL VALIDATION")
print("=" * 45)

print("Customer candidates:", len(customer_candidates))
print("Product candidates:", len(product_candidates))
print("Retention candidates:", len(retention_candidates))

print("\nCustomer source:")
print(customer_candidates["Source"].unique())

print("Product source:")
print(product_candidates["Source"].unique())

print("Retention source:")
print(retention_candidates["Source"].unique())

CANDIDATE RETRIEVAL VALIDATION
Customer candidates: 4
Product candidates: 20
Retention candidates: 4

Customer source:
<StringArray>
['customer_segment']
Length: 1, dtype: str
Product source:
<StringArray>
['product_summary']
Length: 1, dtype: str
Retention source:
<StringArray>
['activity_summary']
Length: 1, dtype: str


## Cross-Encoder Reranker

In [37]:
def rerank_candidates(query, candidates, top_k=5):
    pairs = [
        [query, text]
        for text in candidates["Text"]
    ]

    rerank_scores = reranker.predict(pairs)

    reranked = candidates.copy()
    reranked["Rerank_Score"] = rerank_scores

    reranked = (
        reranked
        .sort_values("Rerank_Score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    reranked["Rerank_Rank"] = range(
        1,
        len(reranked) + 1
    )

    return reranked[
        [
            "Rerank_Rank",
            "Rank",
            "Semantic_Score",
            "Rerank_Score",
            "Source",
            "Document_ID",
            "Text"
        ]
    ]

In [38]:
customer_reranked = rerank_candidates(
    "Which customer segment generates the most revenue?",
    customer_candidates,
    top_k=4
)

customer_reranked

,Rerank_Rank,Rank,Semantic_Score,Rerank_Score,Source,Document_ID,Text
0,1,3,0.733111,8.177780,customer_segment,customer_segment_Very High,"Customer revenue segment Very High contains 1,..."
1,2,2,0.756317,7.965140,customer_segment,customer_segment_High,"Customer revenue segment High contains 1,083 c..."
2,3,4,0.632962,7.537302,customer_segment,customer_segment_Low,"Customer revenue segment Low contains 1,084 cu..."
3,4,1,0.764751,7.521425,customer_segment,customer_segment_Medium,"Customer revenue segment Medium contains 1,084..."


In [39]:
product_reranked = rerank_candidates(
    "Which products are strongest revenue contributors?",
    product_candidates,
    top_k=5
)

product_reranked

,Rerank_Rank,Rank,Semantic_Score,Rerank_Score,Source,Document_ID,Text
0,1,7,0.409420,-4.980296,product_summary,product_summary_22960,Product revenue analysis: JAM MAKING SET WITH ...
1,2,2,0.443205,-5.053413,product_summary,product_summary_23843,"Product revenue analysis: PAPER CRAFT , LITTLE..."
2,3,4,0.418270,-5.223352,product_summary,product_summary_23166,Product revenue analysis: MEDIUM CERAMIC TOP S...
3,4,3,0.435921,-5.438643,product_summary,product_summary_84879,Product revenue analysis: ASSORTED COLOUR BIRD...
4,5,13,0.379837,-5.641366,product_summary,product_summary_82484,Product revenue analysis: WOOD BLACK BOARD ANT...


In [40]:
retention_reranked = rerank_candidates(
    "Which customer groups are at risk of becoming inactive?",
    retention_candidates,
    top_k=4
)

retention_reranked

,Rerank_Rank,Rank,Semantic_Score,Rerank_Score,Source,Document_ID,Text
0,1,2,0.510447,-0.773824,activity_summary,activity_summary_Recently Inactive,Customer activity segment Recently Inactive co...
1,2,1,0.527435,-1.042296,activity_summary,activity_summary_Inactive,Customer activity segment Inactive contains 86...
2,3,3,0.422379,-4.572865,activity_summary,activity_summary_At Risk,Customer activity segment At Risk contains 586...
3,4,4,0.392397,-9.211955,activity_summary,activity_summary_Active,"Customer activity segment Active contains 1,64..."


In [41]:
print("CUSTOMER")
print(customer_reranked[
    ["Rerank_Rank", "Rank", "Semantic_Score", "Rerank_Score", "Document_ID"]
])

print("\nPRODUCT")
print(product_reranked[
    ["Rerank_Rank", "Rank", "Semantic_Score", "Rerank_Score", "Document_ID"]
])

print("\nRETENTION")
print(retention_reranked[
    ["Rerank_Rank", "Rank", "Semantic_Score", "Rerank_Score", "Document_ID"]
])

CUSTOMER
   Rerank_Rank  Rank  Semantic_Score  Rerank_Score                 Document_ID
0            1     3        0.733111      8.177780  customer_segment_Very High
1            2     2        0.756317      7.965140       customer_segment_High
2            3     4        0.632962      7.537302        customer_segment_Low
3            4     1        0.764751      7.521425     customer_segment_Medium

PRODUCT
   Rerank_Rank  Rank  Semantic_Score  Rerank_Score            Document_ID
0            1     7        0.409420     -4.980296  product_summary_22960
1            2     2        0.443205     -5.053413  product_summary_23843
2            3     4        0.418270     -5.223352  product_summary_23166
3            4     3        0.435921     -5.438643  product_summary_84879
4            5    13        0.379837     -5.641366  product_summary_82484

RETENTION
   Rerank_Rank  Rank  Semantic_Score  Rerank_Score  \
0            1     2        0.510447     -0.773824   
1            2     1    

## Compare FAISS vs Reranked Results

In [42]:
print("CUSTOMER SEGMENT RANKING")
print("=" * 50)

print("FAISS semantic ranking:")
print(
    customer_candidates[
        ["Rank", "Semantic_Score", "Document_ID"]
    ].to_string(index=False)
)

print("\nCross-encoder reranking:")
print(
    customer_reranked[
        ["Rerank_Rank", "Rerank_Score", "Document_ID"]
    ].to_string(index=False)
)

CUSTOMER SEGMENT RANKING
FAISS semantic ranking:
 Rank  Semantic_Score                Document_ID
    1        0.764751    customer_segment_Medium
    2        0.756317      customer_segment_High
    3        0.733111 customer_segment_Very High
    4        0.632962       customer_segment_Low

Cross-encoder reranking:
 Rerank_Rank  Rerank_Score                Document_ID
           1      8.177780 customer_segment_Very High
           2      7.965140      customer_segment_High
           3      7.537302       customer_segment_Low
           4      7.521425    customer_segment_Medium


In [43]:
print("PRODUCT RANKING")
print("=" * 50)

print("FAISS semantic ranking:")
print(
    product_candidates[
        ["Rank", "Semantic_Score", "Document_ID"]
    ].head(10).to_string(index=False)
)

print("\nCross-encoder reranking:")
print(
    product_reranked[
        ["Rerank_Rank", "Rerank_Score", "Document_ID"]
    ].to_string(index=False)
)

PRODUCT RANKING
FAISS semantic ranking:
 Rank  Semantic_Score           Document_ID
    1        0.465162 product_summary_22197
    2        0.443205 product_summary_23843
    3        0.435921 product_summary_84879
    4        0.418270 product_summary_23166
    5        0.415534 product_summary_21137
    6        0.414155 product_summary_47566
    7        0.409420 product_summary_22960
    8        0.403034 product_summary_22086
    9        0.401836 product_summary_22386
   10        0.393035 product_summary_23084

Cross-encoder reranking:
 Rerank_Rank  Rerank_Score           Document_ID
           1     -4.980296 product_summary_22960
           2     -5.053413 product_summary_23843
           3     -5.223352 product_summary_23166
           4     -5.438643 product_summary_84879
           5     -5.641366 product_summary_82484


In [44]:
print("RETENTION RANKING")
print("=" * 50)

print("FAISS semantic ranking:")
print(
    retention_candidates[
        ["Rank", "Semantic_Score", "Document_ID"]
    ].to_string(index=False)
)

print("\nCross-encoder reranking:")
print(
    retention_reranked[
        ["Rerank_Rank", "Rerank_Score", "Document_ID"]
    ].to_string(index=False)
)

RETENTION RANKING
FAISS semantic ranking:
 Rank  Semantic_Score                        Document_ID
    1        0.527435          activity_summary_Inactive
    2        0.510447 activity_summary_Recently Inactive
    3        0.422379           activity_summary_At Risk
    4        0.392397            activity_summary_Active

Cross-encoder reranking:
 Rerank_Rank  Rerank_Score                        Document_ID
           1     -0.773824 activity_summary_Recently Inactive
           2     -1.042296          activity_summary_Inactive
           3     -4.572865           activity_summary_At Risk
           4     -9.211955            activity_summary_Active


In [45]:
def show_rank_changes(original, reranked):
    original_rank = original.set_index("Document_ID")["Rank"]
    new_rank = reranked.set_index("Document_ID")["Rerank_Rank"]

    comparison = pd.DataFrame({
        "FAISS_Rank": original_rank,
        "Rerank_Rank": new_rank
    })

    comparison["Rank_Change"] = (
        comparison["FAISS_Rank"] -
        comparison["Rerank_Rank"]
    )

    return comparison.sort_values(
        "Rank_Change",
        ascending=False
    )


print("Customer rank changes:")
print(show_rank_changes(
    customer_candidates,
    customer_reranked
))

print("\nProduct rank changes:")
print(show_rank_changes(
    product_candidates,
    product_reranked
))

print("\nRetention rank changes:")
print(show_rank_changes(
    retention_candidates,
    retention_reranked
))

Customer rank changes:
                            FAISS_Rank  Rerank_Rank  Rank_Change
Document_ID                                                     
customer_segment_Very High           3            1            2
customer_segment_Low                 4            3            1
customer_segment_High                2            2            0
customer_segment_Medium              1            4           -3

Product rank changes:
                        FAISS_Rank  Rerank_Rank  Rank_Change
Document_ID                                                 
product_summary_82484           13          5.0          8.0
product_summary_22960            7          1.0          6.0
product_summary_23166            4          3.0          1.0
product_summary_23843            2          2.0          0.0
product_summary_84879            3          4.0         -1.0
product_summary_21137            5          NaN          NaN
product_summary_22086            8          NaN          NaN
product_summary

## Reranking + Business Signals

In [51]:
# Retrieve a larger semantic candidate pool
product_candidates_20 = retrieve_candidates(
    "Which products are strongest revenue contributors?",
    source="product_summary",
    top_k=20
)

print("Candidate products:", len(product_candidates_20))

product_candidates_20[
    ["Rank", "Semantic_Score", "Document_ID"]
]

Candidate products: 20


,Rank,Semantic_Score,Document_ID
0,1,0.465162,product_summary_22197
1,2,0.443205,product_summary_23843
2,3,0.435921,product_summary_84879
3,4,0.418270,product_summary_23166
4,5,0.415534,product_summary_21137
5,6,0.414155,product_summary_47566
6,7,0.409420,product_summary_22960
7,8,0.403034,product_summary_22086
8,9,0.401836,product_summary_22386
9,10,0.393035,product_summary_23084


In [52]:
product_reranked_20 = rerank_candidates(
    "Which products are strongest revenue contributors?",
    product_candidates_20,
    top_k=20
)

product_reranked_20[
    [
        "Rerank_Rank",
        "Rank",
        "Semantic_Score",
        "Rerank_Score",
        "Document_ID"
    ]
]

,Rerank_Rank,Rank,Semantic_Score,Rerank_Score,Document_ID
0,1,7,0.409420,-4.980296,product_summary_22960
1,2,2,0.443205,-5.053413,product_summary_23843
2,3,4,0.418270,-5.223352,product_summary_23166
3,4,3,0.435921,-5.438643,product_summary_84879
4,5,13,0.379837,-5.641366,product_summary_82484
5,6,20,0.345652,-5.661427,product_summary_79321
6,7,14,0.379477,-5.776855,product_summary_85099B
7,8,19,0.359367,-5.801227,product_summary_23284
8,9,8,0.403034,-5.910734,product_summary_22086
9,10,10,0.393035,-6.043919,product_summary_23084


In [53]:
product_reranked_business = product_reranked_20.merge(
    product_business[
        ["Document_ID", "Total_Revenue"]
    ],
    on="Document_ID",
    how="left"
)

# Normalize reranker score
product_reranked_business["Rerank_Normalized"] = (
    product_reranked_business["Rerank_Score"] -
    product_reranked_business["Rerank_Score"].min()
) / (
    product_reranked_business["Rerank_Score"].max() -
    product_reranked_business["Rerank_Score"].min()
)

# Normalize revenue
product_reranked_business["Revenue_Normalized"] = (
    product_reranked_business["Total_Revenue"] -
    product_reranked_business["Total_Revenue"].min()
) / (
    product_reranked_business["Total_Revenue"].max() -
    product_reranked_business["Total_Revenue"].min()
)

# Combined semantic + business score
product_reranked_business["Business_Score"] = (
    0.4 * product_reranked_business["Rerank_Normalized"] +
    0.6 * product_reranked_business["Revenue_Normalized"]
)

product_business_ranked = (
    product_reranked_business
    .sort_values("Business_Score", ascending=False)
    .reset_index(drop=True)
)

product_business_ranked["Business_Rank"] = (
    range(1, len(product_business_ranked) + 1)
)

product_business_ranked[
    [
        "Business_Rank",
        "Document_ID",
        "Rerank_Score",
        "Total_Revenue",
        "Business_Score"
    ]
].head(10)

,Business_Rank,Document_ID,Rerank_Score,Total_Revenue,Business_Score
0,1,product_summary_23843,-5.053413,168469.60,0.956736
1,2,product_summary_22423,-6.352470,174156.54,0.651470
2,3,product_summary_23166,-5.223352,81700.92,0.536835
3,4,product_summary_85099B,-5.776855,94159.81,0.450341
4,5,product_summary_22960,-4.980296,37082.13,0.404842
5,6,product_summary_85123A,-6.189044,104462.75,0.390380
6,7,product_summary_84879,-5.438643,58927.62,0.383273
7,8,product_summary_79321,-5.661427,54096.36,0.305710
8,9,product_summary_22086,-5.910734,64875.59,0.289188
9,10,product_summary_47566,-6.555111,99445.23,0.275614


In [54]:
actual_top_products = (
    merchandise_products
    .sort_values("Total_Revenue", ascending=False)
    .head(5)
)

expected_ids = [
    f"product_summary_{stock_code}"
    for stock_code in actual_top_products["StockCode"]
]

predicted_ids = product_business_ranked[
    "Document_ID"
].head(5).tolist()

overlap = len(
    set(expected_ids) & set(predicted_ids)
)

print("ACTUAL TOP 5:")
print(expected_ids)

print("\nPREDICTED TOP 5:")
print(predicted_ids)

print("\nTop-5 overlap:", overlap, "/ 5")
print("Business ranking coverage:", f"{overlap / 5:.2%}")

print("\nPredicted products:")
print(
    product_business_ranked[
        [
            "Business_Rank",
            "Document_ID",
            "Total_Revenue",
            "Business_Score"
        ]
    ].head(5).to_string(index=False)
)

ACTUAL TOP 5:
['product_summary_22423', 'product_summary_23843', 'product_summary_85123A', 'product_summary_47566', 'product_summary_85099B']

PREDICTED TOP 5:
['product_summary_23843', 'product_summary_22423', 'product_summary_23166', 'product_summary_85099B', 'product_summary_22960']

Top-5 overlap: 3 / 5
Business ranking coverage: 60.00%

Predicted products:
 Business_Rank            Document_ID  Total_Revenue  Business_Score
             1  product_summary_23843      168469.60        0.956736
             2  product_summary_22423      174156.54        0.651470
             3  product_summary_23166       81700.92        0.536835
             4 product_summary_85099B       94159.81        0.450341
             5  product_summary_22960       37082.13        0.404842


In [55]:

product_business_first = product_reranked_business.copy()

product_business_first["Business_First_Score"] = (
    0.8 * product_business_first["Revenue_Normalized"] +
    0.2 * product_business_first["Rerank_Normalized"]
)

product_business_first = (
    product_business_first
    .sort_values(
        "Business_First_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

product_business_first["Business_First_Rank"] = (
    range(1, len(product_business_first) + 1)
)

print("BUSINESS-FIRST PRODUCT RANKING")
print("=" * 50)

print(
    product_business_first[
        [
            "Business_First_Rank",
            "Document_ID",
            "Total_Revenue",
            "Rerank_Score",
            "Business_First_Score"
        ]
    ].head(10).to_string(index=False)
)

BUSINESS-FIRST PRODUCT RANKING
 Business_First_Rank            Document_ID  Total_Revenue  Rerank_Score  Business_First_Score
                   1  product_summary_23843      168469.60     -5.053413              0.957792
                   2  product_summary_22423      174156.54     -6.352470              0.825735
                   3 product_summary_85123A      104462.75     -6.189044              0.443023
                   4 product_summary_85099B       94159.81     -5.776855              0.435725
                   5  product_summary_23166       81700.92     -5.223352              0.433893
                   6  product_summary_47566       99445.23     -6.555111              0.367485
                   7  product_summary_84879       58927.62     -5.438643              0.274713
                   8  product_summary_22086       64875.59     -5.910734              0.249192
                   9  product_summary_23084       66870.03     -6.043919              0.243824
                  1

In [56]:
expected_ids = [
    f"product_summary_{stock_code}"
    for stock_code in actual_top_products["StockCode"]
]

predicted_ids = product_business_first[
    "Document_ID"
].head(5).tolist()

overlap = len(
    set(expected_ids) & set(predicted_ids)
)

print("Expected top-5:", expected_ids)
print("Predicted top-5:", predicted_ids)

print("\nTop-5 overlap:", overlap, "/ 5")
print("Coverage:", f"{overlap / 5:.2%}")

Expected top-5: ['product_summary_22423', 'product_summary_23843', 'product_summary_85123A', 'product_summary_47566', 'product_summary_85099B']
Predicted top-5: ['product_summary_23843', 'product_summary_22423', 'product_summary_85123A', 'product_summary_85099B', 'product_summary_23166']

Top-5 overlap: 4 / 5
Coverage: 80.00%


## Reranking Evaluation

In [58]:
customer_expected = "customer_segment_Very High"

customer_faiss_top = customer_candidates.iloc[0]["Document_ID"]
customer_rerank_top = customer_reranked.iloc[0]["Document_ID"]

print("CUSTOMER RERANKING")
print("=" * 45)
print("Expected:", customer_expected)
print("FAISS top:", customer_faiss_top)
print("Reranked top:", customer_rerank_top)

print("\nFAISS correct:",
    customer_faiss_top == customer_expected)

print("Reranked correct:",
    customer_rerank_top == customer_expected)

CUSTOMER RERANKING
Expected: customer_segment_Very High
FAISS top: customer_segment_Medium
Reranked top: customer_segment_Very High

FAISS correct: False
Reranked correct: True


In [60]:
retention_expected = "activity_summary_Recently Inactive"

retention_faiss_top = retention_candidates.iloc[0]["Document_ID"]
retention_rerank_top = retention_reranked.iloc[0]["Document_ID"]

print("RETENTION RERANKING")
print("=" * 45)
print("Expected:", retention_expected)
print("FAISS top:", retention_faiss_top)
print("Reranked top:", retention_rerank_top)

print("\nFAISS correct:",
    retention_faiss_top == retention_expected)

print("Reranked correct:",
    retention_rerank_top == retention_expected)

RETENTION RERANKING
Expected: activity_summary_Recently Inactive
FAISS top: activity_summary_Inactive
Reranked top: activity_summary_Recently Inactive

FAISS correct: False
Reranked correct: True


In [62]:
actual_top_5_ids = [
    f"product_summary_{stock_code}"
    for stock_code in actual_top_products["StockCode"]
]

predicted_business_top_5 = (
    product_business_first["Document_ID"]
    .head(5)
    .tolist()
)

product_overlap = len(
    set(actual_top_5_ids) &
    set(predicted_business_top_5)
)

print("PRODUCT BUSINESS RANKING")
print("=" * 45)
print("Expected top-5:", actual_top_5_ids)
print("Predicted top-5:", predicted_business_top_5)

print("\nTop-5 overlap:",
    product_overlap, "/ 5")

print("Coverage:",
    f"{product_overlap / 5:.2%}")

PRODUCT BUSINESS RANKING
Expected top-5: ['product_summary_22423', 'product_summary_23843', 'product_summary_85123A', 'product_summary_47566', 'product_summary_85099B']
Predicted top-5: ['product_summary_23843', 'product_summary_22423', 'product_summary_85123A', 'product_summary_85099B', 'product_summary_23166']

Top-5 overlap: 4 / 5
Coverage: 80.00%


In [64]:
customer_faiss_correct = (
    customer_faiss_top == customer_expected
)

customer_rerank_correct = (
    customer_rerank_top == customer_expected
)

retention_faiss_correct = (
    retention_faiss_top == retention_expected
)

retention_rerank_correct = (
    retention_rerank_top == retention_expected
)

print("EVALUATION SUMMARY")
print("=" * 50)

print(
    "Customer — FAISS:",
    "PASS" if customer_faiss_correct else "FAIL"
)

print(
    "Customer — Cross-Encoder:",
    "PASS" if customer_rerank_correct else "FAIL"
)

print(
    "Retention — FAISS:",
    "PASS" if retention_faiss_correct else "FAIL"
)

print(
    "Retention — Cross-Encoder:",
    "PASS" if retention_rerank_correct else "FAIL"
)

print(
    "Product — Business Top-5 Coverage:",
    f"{product_overlap / 5:.2%}"
)

EVALUATION SUMMARY
Customer — FAISS: FAIL
Customer — Cross-Encoder: PASS
Retention — FAISS: FAIL
Retention — Cross-Encoder: PASS
Product — Business Top-5 Coverage: 80.00%
